In [ ]:

from tdqeq.pipeline import Pipeline
import json

# 2. Build the Pipeline
pipeline = Pipeline(
    dpi=200,
    device='cpu',
    batch_size=4,
    mode='auto',
)

In [ ]:
import requests
response= requests.get("https://download.siliconexpert.com/pdfs2/2025/1/8/15/42/42/220292/roh_/manual/esr-e.pdf")

tables = pipeline.run(response.content, page_range=(0,0))
print(f"Extracted {len(tables)} tables!")

In [ ]:
TDQEQ_OPENAI_API_KEY="AQ.Ab8RN6J6Q7ocr4aZ8_nbGcsDPMKiePVqG5Y5_L3OmG6mhj5Kpw"
TDQEQ_OPENAI_MODEL="models/gemini-3.5-flash-lite"
TDQEQ_OPENAI_BASE_URL="https://generativelanguage.googleapis.com/v1beta/openai/"

import openai
import json_repair
import pandas as pd


client = openai.OpenAI(api_key=TDQEQ_OPENAI_API_KEY, base_url=TDQEQ_OPENAI_BASE_URL)


In [ ]:
SYSTEM_PROMPT= """
You are a deterministic data transformation engine. Your sole task is to convert a list of JSON objects, each containing an HTML table layout, into a highly optimized, mapped, and nested JSON structure.

### CORE DEFINITIONS FOR THIS TASK:
1. **Primary Entity**: The main subject characterizing a logical record (e.g., a Part Number or ID). 
2. **Shared Attributes**: Columns containing data that apply universally to the Primary Entity (often visually represented by cells spanning multiple rows via `rowspan`).
3. **Variant Attributes (1-to-Many)**: Columns containing distinct sub-records that belong to the Primary Entity.

### INPUT FORMAT:
You will receive a JSON array of objects: `[{"html": "...", "page_number": <int>, "confidence_score": <float>}]`

### OUTPUT FORMAT (Mapped & Nested JSON):
You must output a single JSON array containing one object per table (or merged table). Each object must have:
- `page_number`: The integer (or array of integers if merged).
- `confidence_score`: The float (or minimum float if merged).
- `table_title`: A string extracted from the `class` attribute of the `<table>` tag. If none, use `""`.
- `mapping_scratchpad`: A brief string where you explicitly write out your Chain-of-Thought (normalization steps, cross-page merges, and Primary Key selection) before generating data.
- `column_mapping`: A dictionary where keys are sequential IDs (`"c1"`, `"c2"`, `"c3"`, etc.) and values are the exact verbatim column headers derived from the table hierarchy (join multi-tier headers with " / ").
- `data`: An array of objects representing the extracted entities.

**Data Payload Laws (`data` array):**
- Use the `c_` IDs from your `column_mapping` as the keys for Shared Attributes.
- **Strict Data Integrity**: Preserve all extracted values EXACTLY as they appear in the source HTML. Do not strip whitespace, do not abbreviate words, and do not alter casing.
- **Dynamic Positional Matrix (The 1-to-Many Solution)**: If a Primary Entity contains multiple sub-rows (Variant Attributes), do NOT output standard key-value pairs for those nested variables. Instead:
  1. Define an array named `v_cols` containing the specific column IDs that apply to the nested rows (e.g., `["c8", "c9", "c10"]`).
  2. Define an array named `v` containing an array of string arrays, where each inner array represents one sub-row's verbatim cell values in the exact order dictated by `v_cols`.
- **Empty Cells**: Completely empty cells become `"[Blank]"`. Ignore entirely empty rows.
- **No Conversational Filler**: Output only the final JSON array. Do not include markdown code blocks, introductions, or explanations.

---

## PHASE 0 — PRE-PROCESSING (FILTERING & AGGRESSIVE MERGING)
Before analyzing tables individually, aggressively scan the entire input array from start to finish to detect tables split across document breaks.
- **Layout Table Rejection**: If a table contains no relational data (e.g., just page headers/logos), omit it entirely.
- **CRITICAL - Merge Trigger**: If Table B immediately follows Table A AND meets ANY of these conditions, you MUST merge them:
  1. **Title Match**: They share the exact same non-empty `table_title`.
  2. **Orphan Continuation**: Table B lacks a title, has the exact same column count/geometry as Table A, and acts as a semantic continuation of Table A's data.
- **Merge Action**: Logically fuse their HTML rows into a single continuous table. Table B inherits Table A's headers. Output as a single JSON object.

## PHASE 1 — STRUCTURAL MAPPING & NORMALIZATION (Document in `mapping_scratchpad`)
For each logical table, determine and write down:
1. **Grid Normalization (The Matrix Projection)**: Mentally project the raw HTML into a strictly symmetrical 2D grid to resolve structural anomalies and OCR breakage:
   - **Colspan Resolution**: If a cell has `colspan="X"`, explicitly replicate that cell's value `X` times across the adjacent virtual columns.
   - **Sequential Rowspan Unpacking (Merged Text)**: If a cell has `rowspan="Y"` AND contains multiple distinct strings (separated by spaces or line breaks), evaluate if the number of distinct strings matches `Y`. If so, DO NOT replicate the entire cell value. Instead, split the text and assign one unique string to each of the `Y` virtual rows sequentially.
   - **Standard Rowspan Resolution**: If a cell has `rowspan="Y"` and contains only a single logical value, explicitly replicate that cell's value `Y` times down the virtual column.
   - **Orphaned Fragment Stitching (Shattered Rows)**: Scan for fragmented data rows caused by structural document breaks (e.g., cells containing hanging text or abrupt empty sibling cells). Mentally stitch these fragments vertically to the nearest logical parent row before deriving any values.
   - **Jagged Rows (Missing Cells)**: If a row has fewer `<td>` elements than the header columns, implicitly pad the missing trailing cells with `"[Blank]"`.
2. **Header Tree**: Build the hierarchy of header labels and map them to `c_` IDs.
3. **Primary Entity Inference**: Scan all normalized columns to identify the Primary Key using this weighted heuristic:
   - *Positive Signals*: Headers containing `ID`, `Code`, `SKU`, `Ref`, `Part No`, or `No.`. Cell values with alphanumeric patterns or fixed-length strings. Give massive preference to index 0 or 1.
   - *Negative Constraints*: MUST ignore columns representing continuous metrics (`Price`, `Qty`, `Weight`), dates, or booleans.

## PHASE 2 — DERIVING ENTITIES AND VARIANTS (The Universal Span Rule)
Based on the normalized grid, you must strictly classify every column as either a Shared Attribute or a Variant Attribute before extracting data.

1. **Classify Shared Attributes (Root Keys)**: Look at the complete block of rows belonging to a single Primary Entity. IF AND ONLY IF a column contains exactly ONE unique value that spans the ENTIRE block (e.g., a single `rowspan` that covers every sub-row perfectly), it is a Shared Attribute. Assign it directly to the root entity object using its `c_` ID.
2. **Classify Variant Attributes (Matrix Keys)**: If a column changes value at ANY point within the Primary Entity's block (e.g., it contains two different vertically stacked `rowspan` cells, or distinct sub-row values), it is a Variant Attribute. It CANNOT be a root key. You MUST assign its column ID to the `v_cols` array.
3. **Construct the Matrix**: Iterate through every sub-row of the Primary Entity. For every column identified as a Variant Attribute, append its verbatim cell value to the `v` array in the exact order dictated by `v_cols`. Do not duplicate sub-rows unless explicitly unpacked during Phase 1.

## PHASE 3 — SELF-VERIFICATION 
Check each table's result before returning the final JSON:
1. **Coverage**: Every data cell from the normalized grid appears exactly once.
2. **Symmetry**: Ensure that every string array inside `v` contains the exact same number of elements as the `v_cols` array. 
**Escalation**: If verification fails for specific rows (due to ambiguous source structure), completely **DROP** those broken rows from the `data` array so they do not pollute the output. Add a string to the `"note"` key at the root of the table's JSON explaining which rows were dropped and why.
"""

In [ ]:
user_message=f"""
{tables}
"""

In [ ]:
response = client.chat.completions.create(
    model=TDQEQ_OPENAI_MODEL,
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT.strip()},
        {"role": "user", "content": user_message.strip()},
    ],
    temperature=0.0,
)

input_tokens = response.usage.prompt_tokens
output_tokens = response.usage.completion_tokens
total_tokens = response.usage.total_tokens

print(f"Input Tokens: {input_tokens}")
print(f"Output Tokens: {output_tokens}")
print(f"Total Tokens: {total_tokens}")


In [ ]:
content = response.choices[0].message.content
json_data= json_repair.loads(content)

In [ ]:
def unpack_json_to_output(data):
    output = []
    
    for table in data:
        # 1. Grab the original top-level fields
        page_number = table.get("page_number")
        confidence_score = table.get("confidence_score")
        table_title = table.get("table_title")
        column_mapping = table.get("column_mapping", {})
        
        unpacked_rows = []
        
        for row in table.get("data", []):
            # 2. Extract base/shared attributes (any key starting with 'c')
            base_attrs = {k: v for k, v in row.items() if k.startswith("c")}
            
            v_cols = row.get("v_cols", [])
            v_data = row.get("v", [])
            
            if v_data:
                # 3. If variants exist, merge each variant with the base attributes
                for variant_row in v_data:
                    combined = base_attrs.copy()
                    # Map the list of values to their corresponding 'cX' keys
                    for i, col_key in enumerate(v_cols):
                        combined[col_key] = variant_row[i]
                    
                    # 4. Translate 'cX' keys to human-readable names using column_mapping
                    translated_row = {
                        column_mapping.get(k, k): v for k, v in combined.items()
                    }
                    unpacked_rows.append(translated_row)
            else:
                # 5. If no variants exist, just translate the base attributes
                translated_row = {
                    column_mapping.get(k, k): v for k, v in base_attrs.items()
                }
                unpacked_rows.append(translated_row)
        
        # 6. Assemble the final object structure for this table
        output.append({
            "page_number": page_number,
            "confidence_score": confidence_score,
            "table_title": table_title,
            "rows": unpacked_rows
        })
                
    return output


In [ ]:
flat_rows = unpack_json_to_output(json_data)
flat_rows